# Week 6 Lab — ARMA Model Selection and Out-of-Sample Forecasting

**MANG2074 Financial Econometrics 1**

**Objectives**

- Select ARMA orders by AIC/BIC grid search over $p, q \le 5$.
- Produce static (one-step) and dynamic (multi-step) out-of-sample forecasts.
- Evaluate forecasts with RMSE and MAE against a naive benchmark.
- Benchmark against simple exponential smoothing.

**Data**

`../data/ukhp.csv` — monthly UK house prices, 1991–2018. Training sample: to 2015-12. Hold-out: 2016-01 to 2018-03 (27 months).


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
import warnings
warnings.filterwarnings('ignore')

hp = pd.read_csv('../data/ukhp.csv', index_col=0, parse_dates=True).asfreq('MS')
hp['dhp'] = 100 * hp['Average House Price'].pct_change()
dhp = hp['dhp'].dropna()

train = dhp[:'2015-12-01']
test = dhp['2016-01-01':]
print(f"training obs: {len(train)}, hold-out obs: {len(test)}")

## Task 1 — AIC/BIC grid search

Loop over all $(p, q)$ with $0 \le p, q \le 5$, fit `ARIMA(dhp, order=(p,0,q))` on the **full sample**, and store AIC and BIC in two 6×6 DataFrames (rows = p, columns = q). Which $(p,q)$ minimises each criterion? (This will take ~10 seconds; ignore convergence warnings.)

In [ ]:
aic = pd.DataFrame(np.nan, index=range(6), columns=range(6))
bic = pd.DataFrame(np.nan, index=range(6), columns=range(6))

for p in range(6):
    for q in range(6):
        try:
            res = ARIMA(dhp, order=(p, 0, q)).fit()
            # YOUR CODE HERE: store res.aic and res.bic
        except Exception:
            pass

# YOUR CODE HERE: print the tables and find the (p,q) minimising each
# Hint: aic.stack().idxmin()


## Task 2 — Estimate the chosen models on the training sample only

Re-estimate the BIC choice (AR(2)) and the AIC choice (ARMA(4,2)) using **only the training data** (to Dec 2015). Why must the forecasting model not see the hold-out data, even at the selection stage?

In [ ]:
res_bic = ARIMA(train, order=(2, 0, 0)).fit()
# YOUR CODE HERE: res_aic for ARMA(4,2); print res_bic.summary()


## Task 3 — Static (one-step-ahead) forecasts

Use `res.apply(dhp)` to carry the *training-sample parameters* onto the full data without re-estimating, then `​.predict(start='2016-01-01', end='2018-03-01', dynamic=False)`. Plot forecasts vs actuals. Each forecast here uses actual data up to the previous month.

In [ ]:
applied_bic = res_bic.apply(dhp)   # same parameters, full data, no re-estimation
static_fc = applied_bic.predict(start='2016-01-01', end='2018-03-01', dynamic=False)

# YOUR CODE HERE: plot test (actuals) and static_fc on one figure


## Task 4 — Dynamic (multi-step) forecasts

Repeat with `dynamic=True` (all forecasts from the 2016-01 origin use predicted values in place of actuals). Plot actuals, static and dynamic forecasts together. What value does the dynamic forecast converge to, and why?

In [ ]:
dynamic_fc = applied_bic.predict(start='2016-01-01', end='2018-03-01', dynamic=True)
# YOUR CODE HERE: combined plot


## Task 5 — RMSE and MAE

Write functions `rmse(f, y)` and `mae(f, y)` and score: static AR(2), dynamic AR(2), static ARMA(4,2), and the naive benchmark (forecast = training-sample mean every month). Put results in a table.

In [ ]:
def rmse(f, y):
    # YOUR CODE HERE
    pass

def mae(f, y):
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: score all four forecast sets


## Task 6 — Simple exponential smoothing

Fit `SimpleExpSmoothing(train).fit()` (it chooses $\alpha$ by maximum likelihood — report it), produce forecasts for the 27 hold-out months with `.forecast(len(test))`, and add its RMSE/MAE to your table. Note SES out-of-sample forecasts are flat — why?

In [ ]:
ses = SimpleExpSmoothing(train).fit()
# YOUR CODE HERE: alpha, forecasts (remember to align the index: ses_fc.index = test.index), scores


## Task 7 — Verdict

Write 5–8 sentences for the building society: which forecasting approach do you recommend and why? Mention (i) static vs dynamic, (ii) whether the ARMA beats the naive mean and SES, (iii) the limits of a 27-month evaluation window.

*YOUR ANSWER HERE*